# Smart Teen Community - Progress Demonstration

Preliminary Colab application for **M1-M5** and **BA1**.

Run the cells from top to bottom. The notebook follows the Lab 5/6 pattern:
Oracle connection + SQL DML + `ipywidgets` + `google.colab.widgets.TabBar`.

Assumptions used in this draft:
- Member usernames and bank-analyst usernames are primary keys.
- A teenager is treated as age 13-19 inclusive (confirm this business rule).
- New members start with `MEMBER_STATUS = 'PENDING_SURVEY'`.
- Completing the first goal survey changes the status to `ACTIVE`.
- New posts start with `POST_STATUS = 'PENDING_APPROVAL'`.
- Passwords are compared as stored because the current course schema contains a
  `PASSWORD` column. For a production system, store password hashes instead.
- `MAX(id) + 1` is acceptable only for this single-user classroom demo. Use
  Oracle sequences/identity columns for the final concurrent application.



In [ ]:
# Install the same Oracle driver used in Lab 5 and Lab 6.
!pip -q install oracledb --upgrade



In [ ]:
import re
from datetime import date, datetime
from getpass import getpass

import oracledb
import ipywidgets as widgets
from IPython.display import display
from google.colab.widgets import TabBar


# Do not hard-code the project password in a shared notebook.
HOST_NAME = "imz409.ust.hk"
PORT_NUMBER = 1521
SERVICE_NAME = "imz409"
DB_USERNAME = input("Oracle username: ").strip()
DB_PASSWORD = getpass("Oracle password: ")

dsn = oracledb.makedsn(HOST_NAME, PORT_NUMBER, service_name=SERVICE_NAME)
conn = oracledb.connect(user=DB_USERNAME, password=DB_PASSWORD, dsn=dsn)
cursor = conn.cursor()
print("Connected to Oracle.")



## Verify the deployed schema

This catches differences between the latest ERD and older generated SQL before
the demonstration starts.



In [ ]:
EXPECTED_COLUMNS = {
    "MEMBER": {
        "USERNAME", "PASSWORD", "HKID", "MOBILE", "MEMBER_NAME", "EMAIL",
        "ACCOUNT_NO", "DATE_OF_BIRTH", "GENDER", "SALARY_RANGE",
        "MEMBER_STATUS", "REGISTER_DATETIME", "LAST_LOGIN_DATETIME",
        "LAST_LOGOUT_DATETIME",
    },
    "GOAL": {
        "GOAL_ID", "GOAL_DESCRIPTION", "GOAL_AMOUNT", "RISK_APPETITE",
        "TIME_TO_GOAL", "GOAL_REMARK", "SURVEY_DATETIME", "MEMBER_USERNAME",
    },
    "TRANSACTIONS": {
        "T_ID", "TRANSACTION_DATETIME", "TRANSACTION_AMOUNT",
        "TRANSACTION_REMARK", "TRANSACTION_CATEGORY", "TRANSACTION_TYPE",
        "MEMBER_USERNAME",
    },
    "POST": {
        "P_ID", "CONTENT", "TITLE", "POST_STATUS", "SUBMITTED_DATETIME",
        "REVIEW_DATETIME", "PUBLISHED_DATETIME", "REJECTION_JUSTIFICATION",
        "BANK_ANALYST_BA_USERNAME", "MEMBER_USERNAME",
    },
    "BANK_ANALYST": {"BA_USERNAME", "BA_NAME", "EMAIL", "PASSWORD"},
}

cursor.execute("""
    SELECT table_name, column_name
    FROM user_tab_columns
    WHERE table_name IN ('MEMBER', 'GOAL', 'TRANSACTIONS', 'POST', 'BANK_ANALYST')
    ORDER BY table_name, column_id
""")
actual_columns = {}
for table_name, column_name in cursor:
    actual_columns.setdefault(table_name, set()).add(column_name)

schema_ok = True
for table_name, expected in EXPECTED_COLUMNS.items():
    actual = actual_columns.get(table_name, set())
    missing = sorted(expected - actual)
    if missing:
        schema_ok = False
        print(f"{table_name}: missing {missing}")
    else:
        print(f"{table_name}: OK")

if not schema_ok:
    raise RuntimeError(
        "The deployed tables do not match the latest ERD. Fix the schema or "
        "update EXPECTED_COLUMNS before running the app."
    )



## Shared database and UI helpers



In [ ]:
session = {"role": None, "username": None}


def fetch_one(sql, binds=None):
    cursor.execute(sql, binds or {})
    return cursor.fetchone()


def execute_and_commit(sql, binds=None):
    try:
        cursor.execute(sql, binds or {})
        conn.commit()
    except Exception:
        conn.rollback()
        raise


def next_demo_id(table_name, id_column, username=None):
    """Classroom-only ID generation; table/column names come from fixed code."""
    allowed = {
        ("GOAL", "GOAL_ID"),
        ("TRANSACTIONS", "T_ID"),
        ("POST", "P_ID"),
    }
    if (table_name, id_column) not in allowed:
        raise ValueError("Unsupported ID target")
    if table_name == "GOAL":
        row = fetch_one(
            "SELECT NVL(MAX(GOAL_ID), 0) + 1 FROM GOAL WHERE MEMBER_USERNAME = :username",
            {"username": username},
        )
    else:
        row = fetch_one(f"SELECT NVL(MAX({id_column}), 0) + 1 FROM {table_name}")
    return int(row[0])


def server_age(dob):
    row = fetch_one(
        "SELECT TRUNC(MONTHS_BETWEEN(TRUNC(SYSDATE), :dob) / 12) FROM DUAL",
        {"dob": dob},
    )
    return int(row[0])


def valid_email(value):
    return bool(re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", value or ""))


def as_date(value):
    if value is None:
        return None
    return value.date() if isinstance(value, datetime) else value


def set_message(output, message, error=False):
    output.clear_output()
    with output:
        colour = "#b00020" if error else "#137333"
        display(widgets.HTML(f"<span style='color:{colour}'>{message}</span>"))


def require_member(output):
    if session["role"] != "MEMBER" or not session["username"]:
        set_message(output, "Please log in as a member first.", True)
        return False
    return True


def require_active_member(output):
    if not require_member(output):
        return False
    row = fetch_one(
        "SELECT MEMBER_STATUS FROM MEMBER WHERE USERNAME = :username",
        {"username": session["username"]},
    )
    if not row or row[0] != "ACTIVE":
        set_message(output, "Complete the financial-goal survey before using this function.", True)
        return False
    return True


def require_analyst(output):
    if session["role"] != "BANK_ANALYST" or not session["username"]:
        set_message(output, "Please log in as a bank analyst first.", True)
        return False
    return True



## Application pages



In [ ]:
TAB_NAMES = [
    "Login",
    "Register (M1)",
    "Member Main (M2)",
    "Goal Survey (M3)",
    "Record Income/Expense (M4)",
    "Submit Post (M5)",
    "Analyst Main (BA1)",
]
tabs = TabBar(TAB_NAMES)


def show_page(name):
    with tabs.output_to(name):
        display()


def logout(_=None):
    if session["role"] == "MEMBER" and session["username"]:
        execute_and_commit(
            "UPDATE MEMBER SET LAST_LOGOUT_DATETIME = SYSDATE WHERE USERNAME = :username",
            {"username": session["username"]},
        )
    session.update(role=None, username=None)
    login_password.value = ""
    set_message(login_output, "Logged out.")
    show_page("Login")


# ------------------------- Login -------------------------
with tabs.output_to("Login"):
    login_role = widgets.Dropdown(
        options=[("Member", "MEMBER"), ("Bank analyst", "BANK_ANALYST")],
        description="Role:",
    )
    login_username = widgets.Text(description="Username:")
    login_password = widgets.Password(description="Password:")
    login_button = widgets.Button(description="Log in", button_style="primary")
    register_page_button = widgets.Button(description="Register as member")
    login_output = widgets.Output()

    def on_login(_):
        username = login_username.value.strip()
        password = login_password.value
        if not username or not password:
            set_message(login_output, "Enter both username and password.", True)
            return

        if login_role.value == "MEMBER":
            row = fetch_one(
                "SELECT USERNAME FROM MEMBER WHERE USERNAME = :username AND PASSWORD = :password",
                {"username": username, "password": password},
            )
            if not row:
                set_message(login_output, "Invalid member username or password.", True)
                return

            execute_and_commit(
                "UPDATE MEMBER SET LAST_LOGIN_DATETIME = SYSDATE WHERE USERNAME = :username",
                {"username": username},
            )
            session.update(role="MEMBER", username=username)
            load_member_details()
            set_message(login_output, "Member login successful.")
            show_page("Member Main (M2)")
        else:
            row = fetch_one(
                """SELECT BA_USERNAME FROM BANK_ANALYST
                   WHERE BA_USERNAME = :username AND PASSWORD = :password""",
                {"username": username, "password": password},
            )
            if not row:
                set_message(login_output, "Invalid analyst username or password.", True)
                return
            session.update(role="BANK_ANALYST", username=username)
            load_analyst_details()
            set_message(login_output, "Bank analyst login successful.")
            show_page("Analyst Main (BA1)")

    login_button.on_click(on_login)
    register_page_button.on_click(lambda _: show_page("Register (M1)"))
    display(widgets.VBox([
        widgets.HTML("<h3>Smart Teen Community</h3>"),
        login_role, login_username, login_password,
        widgets.HBox([login_button, register_page_button]),
        login_output,
    ]))


# ------------------------- M1: Registration -------------------------
with tabs.output_to("Register (M1)"):
    reg_username = widgets.Text(description="Username:")
    reg_password = widgets.Password(description="Password:")
    reg_hkid = widgets.Text(description="HKID:")
    reg_mobile = widgets.Text(description="Mobile:")
    reg_name = widgets.Text(description="Name:")
    reg_email = widgets.Text(description="Email:")
    reg_account = widgets.Text(description="Account no.:")
    reg_dob = widgets.DatePicker(description="Date of birth:")
    reg_gender = widgets.Dropdown(
        options=[None, "Male", "Female", "Non-binary", "Prefer not to say"],
        description="Gender:",
    )
    reg_salary = widgets.Dropdown(
        options=[None, "No income", "Below 5,000", "5,000-9,999", "10,000-19,999", "20,000+"],
        description="Salary range:",
    )
    reg_submit = widgets.Button(description="Create account", button_style="success")
    reg_back = widgets.Button(description="Back to login")
    reg_output = widgets.Output()

    def on_register(_):
        values = {
            "username": reg_username.value.strip(),
            "password": reg_password.value,
            "hkid": reg_hkid.value.strip().upper(),
            "mobile": reg_mobile.value.strip(),
            "member_name": reg_name.value.strip(),
            "email": reg_email.value.strip().lower(),
            "account_no": reg_account.value.strip(),
            "date_of_birth": reg_dob.value,
            "gender": reg_gender.value,
            "salary_range": reg_salary.value,
        }
        if any(v in (None, "") for v in values.values()):
            set_message(reg_output, "Complete every registration field.", True)
            return
        length_limits = {
            "username": 20, "password": 100, "hkid": 9,
            "mobile": 20, "member_name": 100, "email": 200,
            "account_no": 30, "gender": 20, "salary_range": 50,
        }
        overlong = next(
            (field for field, limit in length_limits.items()
             if len(values[field]) > limit),
            None,
        )
        if overlong:
            set_message(
                reg_output,
                f"{overlong.replace('_', ' ').title()} exceeds the ERD length.",
                True,
            )
            return
        if not valid_email(values["email"]):
            set_message(reg_output, "Enter a valid email address.", True)
            return
        age = server_age(values["date_of_birth"])
        if age < 13 or age > 19:
            set_message(reg_output, "Registration is limited to teenagers aged 13-19.", True)
            return

        duplicate = fetch_one(
            """SELECT COUNT(*) FROM MEMBER
               WHERE USERNAME = :username OR HKID = :hkid
                  OR EMAIL = :email OR ACCOUNT_NO = :account_no""",
            {
                "username": values["username"],
                "hkid": values["hkid"],
                "email": values["email"],
                "account_no": values["account_no"],
            },
        )[0]
        if duplicate:
            set_message(reg_output, "Username, HKID, email, or account number already exists.", True)
            return

        try:
            execute_and_commit(
                """INSERT INTO MEMBER (
                       USERNAME, PASSWORD, HKID, MOBILE, MEMBER_NAME, EMAIL,
                       ACCOUNT_NO, DATE_OF_BIRTH, GENDER, SALARY_RANGE,
                       MEMBER_STATUS, REGISTER_DATETIME
                   ) VALUES (
                       :username, :password, :hkid, :mobile, :member_name, :email,
                       :account_no, :date_of_birth, :gender, :salary_range,
                       'PENDING_SURVEY', SYSDATE
                   )""",
                values,
            )
            set_message(reg_output, "Account created. You may now log in.")
        except oracledb.Error as exc:
            set_message(reg_output, f"Registration failed: {exc}", True)

    reg_submit.on_click(on_register)
    reg_back.on_click(lambda _: show_page("Login"))
    display(widgets.VBox([
        widgets.HTML("<h3>M1 - Member registration</h3>"),
        reg_username, reg_password, reg_hkid, reg_mobile, reg_name, reg_email,
        reg_account, reg_dob, reg_gender, reg_salary,
        widgets.HBox([reg_submit, reg_back]), reg_output,
    ]))


# ------------------------- M2: Member main/details -------------------------
with tabs.output_to("Member Main (M2)"):
    member_username = widgets.Text(description="Username:", disabled=True)
    member_hkid = widgets.Text(description="HKID:", disabled=True)
    member_mobile = widgets.Text(description="Mobile:")
    member_name = widgets.Text(description="Name:")
    member_email = widgets.Text(description="Email:")
    member_account = widgets.Text(description="Account no.:")
    member_dob = widgets.DatePicker(description="Date of birth:")
    member_gender = widgets.Dropdown(
        options=["Male", "Female", "Non-binary", "Prefer not to say"],
        description="Gender:",
    )
    member_salary = widgets.Dropdown(
        options=["No income", "Below 5,000", "5,000-9,999", "10,000-19,999", "20,000+"],
        description="Salary range:",
    )
    member_status = widgets.Text(description="Status:", disabled=True)
    member_update = widgets.Button(description="Save changes", button_style="success")
    goal_page = widgets.Button(description="Goal survey")
    transaction_page = widgets.Button(description="Record income/expense")
    post_page = widgets.Button(description="Submit post")
    member_logout = widgets.Button(description="Log out", button_style="warning")
    member_output = widgets.Output()

    def load_member_details():
        if session["role"] != "MEMBER":
            return
        row = fetch_one(
            """SELECT USERNAME, HKID, MOBILE, MEMBER_NAME, EMAIL, ACCOUNT_NO,
                      DATE_OF_BIRTH, GENDER, SALARY_RANGE, MEMBER_STATUS
               FROM MEMBER WHERE USERNAME = :username""",
            {"username": session["username"]},
        )
        if row:
            (member_username.value, member_hkid.value, member_mobile.value,
             member_name.value, member_email.value, member_account.value,
             dob, member_gender.value, member_salary.value,
             member_status.value) = row
            member_dob.value = as_date(dob)
            survey_completed = (member_status.value == "ACTIVE")
            transaction_page.disabled = not survey_completed
            post_page.disabled = not survey_completed
            if survey_completed:
                set_message(member_output, "Welcome back.")
            else:
                set_message(
                    member_output,
                    "Reminder: complete the financial-goal survey before using other functions.",
                    True,
                )

    def on_member_update(_):
        if not require_member(member_output):
            return
        if not all([
            member_mobile.value.strip(), member_name.value.strip(),
            member_email.value.strip(), member_account.value.strip(),
            member_dob.value, member_gender.value, member_salary.value,
        ]):
            set_message(member_output, "Complete all editable personal details.", True)
            return
        if not valid_email(member_email.value.strip()):
            set_message(member_output, "Enter a valid email address.", True)
            return
        if any([
            len(member_mobile.value.strip()) > 20,
            len(member_name.value.strip()) > 100,
            len(member_email.value.strip()) > 200,
            len(member_account.value.strip()) > 30,
        ]):
            set_message(member_output, "One or more details exceed the ERD length.", True)
            return
        if not 13 <= server_age(member_dob.value) <= 19:
            set_message(member_output, "Date of birth must represent age 13-19.", True)
            return
        duplicate = fetch_one(
            """SELECT COUNT(*) FROM MEMBER
               WHERE USERNAME <> :username
                 AND (EMAIL = :email OR ACCOUNT_NO = :account_no)""",
            {
                "username": session["username"],
                "email": member_email.value.strip().lower(),
                "account_no": member_account.value.strip(),
            },
        )[0]
        if duplicate:
            set_message(member_output, "Email or account number belongs to another member.", True)
            return
        try:
            execute_and_commit(
                """UPDATE MEMBER
                   SET MOBILE = :mobile, MEMBER_NAME = :member_name,
                       EMAIL = :email, ACCOUNT_NO = :account_no,
                       DATE_OF_BIRTH = :dob, GENDER = :gender,
                       SALARY_RANGE = :salary_range
                   WHERE USERNAME = :username""",
                {
                    "mobile": member_mobile.value.strip(),
                    "member_name": member_name.value.strip(),
                    "email": member_email.value.strip().lower(),
                    "account_no": member_account.value.strip(),
                    "dob": member_dob.value,
                    "gender": member_gender.value,
                    "salary_range": member_salary.value,
                    "username": session["username"],
                },
            )
            set_message(member_output, "Personal details updated.")
        except oracledb.Error as exc:
            set_message(member_output, f"Update failed: {exc}", True)

    member_update.on_click(on_member_update)
    goal_page.on_click(lambda _: show_page("Goal Survey (M3)"))
    transaction_page.on_click(lambda _: show_page("Record Income/Expense (M4)"))
    post_page.on_click(lambda _: show_page("Submit Post (M5)"))
    member_logout.on_click(logout)
    display(widgets.VBox([
        widgets.HTML("<h3>M2 - Member main page and personal details</h3>"),
        member_username, member_hkid, member_mobile, member_name, member_email,
        member_account, member_dob, member_gender, member_salary, member_status,
        member_update,
        widgets.HBox([goal_page, transaction_page, post_page]),
        member_logout, member_output,
    ]))


# ------------------------- M3: Static goal survey -------------------------
with tabs.output_to("Goal Survey (M3)"):
    goal_description = widgets.Textarea(description="Goal:")
    goal_amount = widgets.BoundedFloatText(
        description="Target amount:", min=0, max=9999999999.99, step=0.01
    )
    goal_time = widgets.Dropdown(
        options=[None, "Within 1 year", "1-3 years", "3-5 years", "More than 5 years"],
        description="Time:",
    )
    goal_risk = widgets.RadioButtons(
        options=["Low", "Medium", "High"], description="Risk appetite:"
    )
    goal_remark = widgets.Textarea(description="Remark:")
    goal_submit = widgets.Button(description="Submit survey", button_style="success")
    goal_back = widgets.Button(description="Back to member main")
    goal_output = widgets.Output()

    def on_goal_submit(_):
        if not require_member(goal_output):
            return
        if not all([
            goal_description.value.strip(), goal_amount.value > 0,
            goal_time.value, goal_risk.value,
        ]):
            set_message(goal_output, "Complete all required survey answers.", True)
            return
        if (len(goal_description.value.strip()) > 500 or
                len(goal_remark.value.strip()) > 500):
            set_message(goal_output, "Goal description or remark exceeds 500 characters.", True)
            return
        goal_id = next_demo_id("GOAL", "GOAL_ID", session["username"])
        try:
            cursor.execute(
                """INSERT INTO GOAL (
                       GOAL_ID, GOAL_DESCRIPTION, GOAL_AMOUNT, RISK_APPETITE,
                       TIME_TO_GOAL, GOAL_REMARK, SURVEY_DATETIME, MEMBER_USERNAME
                   ) VALUES (
                       :goal_id, :description, :amount, :risk, :time_to_goal,
                       :remark, SYSDATE, :username
                   )""",
                {
                    "goal_id": goal_id,
                    "description": goal_description.value.strip(),
                    "amount": goal_amount.value,
                    "risk": goal_risk.value,
                    "time_to_goal": goal_time.value,
                    "remark": goal_remark.value.strip() or None,
                    "username": session["username"],
                },
            )
            cursor.execute(
                "UPDATE MEMBER SET MEMBER_STATUS = 'ACTIVE' WHERE USERNAME = :username",
                {"username": session["username"]},
            )
            conn.commit()
            member_status.value = "ACTIVE"
            transaction_page.disabled = False
            post_page.disabled = False
            set_message(goal_output, f"Financial goal saved as goal {goal_id}.")
        except Exception as exc:
            conn.rollback()
            set_message(goal_output, f"Survey submission failed: {exc}", True)

    goal_submit.on_click(on_goal_submit)
    goal_back.on_click(lambda _: show_page("Member Main (M2)"))
    display(widgets.VBox([
        widgets.HTML("<h3>M3 - Static financial-goal survey</h3>"),
        goal_description, goal_amount, goal_time, goal_risk, goal_remark,
        widgets.HBox([goal_submit, goal_back]), goal_output,
    ]))


# ------------------------- M4: Daily income/expense -------------------------
with tabs.output_to("Record Income/Expense (M4)"):
    transaction_type = widgets.RadioButtons(
        options=["EXPENSE", "INCOME"], description="Type:"
    )
    transaction_amount = widgets.BoundedFloatText(
        description="Amount:", min=0, max=9999999999.99, step=0.01
    )
    transaction_category = widgets.Dropdown(
        options=["Gaming", "Fast Food", "Fashion", "Bus Fare", "Education", "Other"],
        description="Category:",
    )
    transaction_remark = widgets.Textarea(description="Remark:")
    transaction_submit = widgets.Button(description="Save entry", button_style="success")
    transaction_back = widgets.Button(description="Back to member main")
    transaction_output = widgets.Output()

    def refresh_transaction_categories(change=None):
        if transaction_type.value == "INCOME":
            transaction_category.options = [
                "Allowance", "Part-time Job", "Gift", "Investment", "Other Income"
            ]
        else:
            transaction_category.options = [
                "Gaming", "Fast Food", "Fashion", "Bus Fare", "Education", "Other"
            ]

    transaction_type.observe(refresh_transaction_categories, names="value")

    def on_transaction_submit(_):
        if not require_active_member(transaction_output):
            return
        if transaction_amount.value <= 0:
            set_message(transaction_output, "Amount must be greater than zero.", True)
            return
        if not transaction_category.value or not transaction_remark.value.strip():
            set_message(transaction_output, "Category and remark are required by the case.", True)
            return
        if len(transaction_remark.value.strip()) > 500:
            set_message(transaction_output, "Remark exceeds 500 characters.", True)
            return
        transaction_id = next_demo_id("TRANSACTIONS", "T_ID")
        try:
            execute_and_commit(
                """INSERT INTO TRANSACTIONS (
                       T_ID, TRANSACTION_DATETIME, TRANSACTION_AMOUNT,
                       TRANSACTION_REMARK, TRANSACTION_CATEGORY,
                       TRANSACTION_TYPE, MEMBER_USERNAME
                   ) VALUES (
                       :transaction_id, SYSDATE, :amount, :remark,
                       :category, :transaction_type, :username
                   )""",
                {
                    "transaction_id": transaction_id,
                    "amount": transaction_amount.value,
                    "remark": transaction_remark.value.strip(),
                    "category": transaction_category.value,
                    "transaction_type": transaction_type.value,
                    "username": session["username"],
                },
            )
            set_message(transaction_output, f"Entry {transaction_id} saved using server time.")
        except Exception as exc:
            set_message(transaction_output, f"Entry failed: {exc}", True)

    transaction_submit.on_click(on_transaction_submit)
    transaction_back.on_click(lambda _: show_page("Member Main (M2)"))
    display(widgets.VBox([
        widgets.HTML("<h3>M4 - Record daily expense or income</h3>"),
        transaction_type, transaction_amount, transaction_category,
        transaction_remark,
        widgets.HBox([transaction_submit, transaction_back]), transaction_output,
    ]))


# ------------------------- M5: Submit post -------------------------
with tabs.output_to("Submit Post (M5)"):
    post_title = widgets.Text(description="Title:")
    post_content = widgets.Textarea(
        description="Content:", layout=widgets.Layout(width="700px", height="180px")
    )
    post_submit = widgets.Button(description="Submit post", button_style="success")
    post_back = widgets.Button(description="Back to member main")
    post_output = widgets.Output()

    def on_post_submit(_):
        if not require_active_member(post_output):
            return
        title = post_title.value.strip()
        content = post_content.value.strip()
        if not title or not content:
            set_message(post_output, "Enter both a title and content.", True)
            return
        if len(title) > 200 or len(content) > 2000:
            set_message(post_output, "Title or content exceeds the ERD length.", True)
            return
        post_id = next_demo_id("POST", "P_ID")
        try:
            execute_and_commit(
                """INSERT INTO POST (
                       P_ID, CONTENT, TITLE, POST_STATUS,
                       SUBMITTED_DATETIME, MEMBER_USERNAME
                   ) VALUES (
                       :post_id, :content, :title, 'PENDING_APPROVAL',
                       SYSDATE, :username
                   )""",
                {
                    "post_id": post_id,
                    "content": content,
                    "title": title,
                    "username": session["username"],
                },
            )
            set_message(post_output, f"Post {post_id} submitted for bank-analyst review.")
        except Exception as exc:
            set_message(post_output, f"Post submission failed: {exc}", True)

    post_submit.on_click(on_post_submit)
    post_back.on_click(lambda _: show_page("Member Main (M2)"))
    display(widgets.VBox([
        widgets.HTML("<h3>M5 - Submit a community post</h3>"),
        post_title, post_content,
        widgets.HBox([post_submit, post_back]), post_output,
    ]))


# ------------------------- BA1: Analyst personal details -------------------------
with tabs.output_to("Analyst Main (BA1)"):
    analyst_username = widgets.Text(description="Username:", disabled=True)
    analyst_name = widgets.Text(description="Name:")
    analyst_email = widgets.Text(description="Email:")
    analyst_update = widgets.Button(description="Save details", button_style="success")
    analyst_current_password = widgets.Password(description="Current pw:")
    analyst_new_password = widgets.Password(description="New pw:")
    analyst_confirm_password = widgets.Password(description="Confirm pw:")
    analyst_change_password = widgets.Button(description="Change password", button_style="warning")
    analyst_logout = widgets.Button(description="Log out")
    analyst_output = widgets.Output()

    def load_analyst_details():
        if session["role"] != "BANK_ANALYST":
            return
        row = fetch_one(
            """SELECT BA_USERNAME, BA_NAME, EMAIL
               FROM BANK_ANALYST WHERE BA_USERNAME = :username""",
            {"username": session["username"]},
        )
        if row:
            analyst_username.value, analyst_name.value, analyst_email.value = row

    def on_analyst_update(_):
        if not require_analyst(analyst_output):
            return
        if not analyst_name.value.strip() or not valid_email(analyst_email.value.strip()):
            set_message(analyst_output, "Enter a name and valid email.", True)
            return
        if len(analyst_name.value.strip()) > 100 or len(analyst_email.value.strip()) > 200:
            set_message(analyst_output, "Name or email exceeds the ERD length.", True)
            return
        try:
            execute_and_commit(
                """UPDATE BANK_ANALYST
                   SET BA_NAME = :name, EMAIL = :email
                   WHERE BA_USERNAME = :username""",
                {
                    "name": analyst_name.value.strip(),
                    "email": analyst_email.value.strip().lower(),
                    "username": session["username"],
                },
            )
            set_message(analyst_output, "Analyst personal details updated.")
        except oracledb.Error as exc:
            set_message(analyst_output, f"Update failed: {exc}", True)

    def on_password_change(_):
        if not require_analyst(analyst_output):
            return
        if not analyst_current_password.value or not analyst_new_password.value:
            set_message(analyst_output, "Enter the current and new passwords.", True)
            return
        if analyst_new_password.value != analyst_confirm_password.value:
            set_message(analyst_output, "New passwords do not match.", True)
            return
        if len(analyst_new_password.value) > 100:
            set_message(analyst_output, "New password exceeds the ERD length.", True)
            return
        row = fetch_one(
            """SELECT 1 FROM BANK_ANALYST
               WHERE BA_USERNAME = :username AND PASSWORD = :password""",
            {
                "username": session["username"],
                "password": analyst_current_password.value,
            },
        )
        if not row:
            set_message(analyst_output, "Current password is incorrect.", True)
            return
        execute_and_commit(
            """UPDATE BANK_ANALYST SET PASSWORD = :new_password
               WHERE BA_USERNAME = :username""",
            {
                "new_password": analyst_new_password.value,
                "username": session["username"],
            },
        )
        analyst_current_password.value = ""
        analyst_new_password.value = ""
        analyst_confirm_password.value = ""
        set_message(analyst_output, "Password changed successfully.")

    analyst_update.on_click(on_analyst_update)
    analyst_change_password.on_click(on_password_change)
    analyst_logout.on_click(logout)
    display(widgets.VBox([
        widgets.HTML("<h3>BA1 - Bank analyst personal information</h3>"),
        analyst_username, analyst_name, analyst_email, analyst_update,
        widgets.HTML("<hr><b>Change password</b>"),
        analyst_current_password, analyst_new_password, analyst_confirm_password,
        analyst_change_password, analyst_logout, analyst_output,
    ]))


show_page("Login")



## Demonstration checklist

1. Register a new teenage member (M1).
2. Log in as that member, show details on the main page, and modify one field (M2).
3. Complete the goal survey and show that `MEMBER_STATUS` becomes `ACTIVE` (M3).
4. Insert one expense and one income entry (M4).
5. Submit a post and show its initial `PENDING_APPROVAL` status in Oracle (M5).
6. Log in as a preloaded bank analyst, modify the analyst's details, and change password (BA1).
7. Call `show_demo_evidence(member_username, analyst_username)` below, or run equivalent `SELECT` statements in SQL Developer, to prove every demonstrated action was committed.
8. Prove the analyst password change by logging out and logging in with the new password; the verification output intentionally never displays passwords.



In [ ]:
def show_demo_evidence(member_username, analyst_username=None):
    """Print committed demo records without exposing passwords or full member data."""
    member_username = member_username.strip()
    if not member_username:
        raise ValueError("Enter the demonstrated member username.")

    checks = [
        ("M1-M3 member", """
            SELECT USERNAME, MEMBER_NAME, EMAIL, MEMBER_STATUS,
                   REGISTER_DATETIME, LAST_LOGIN_DATETIME, LAST_LOGOUT_DATETIME
            FROM MEMBER WHERE USERNAME = :username
        """),
        ("M3 goals", """
            SELECT GOAL_ID, GOAL_DESCRIPTION, GOAL_AMOUNT, RISK_APPETITE,
                   TIME_TO_GOAL, SURVEY_DATETIME
            FROM GOAL WHERE MEMBER_USERNAME = :username
            ORDER BY GOAL_ID
        """),
        ("M4 transactions", """
            SELECT T_ID, TRANSACTION_DATETIME, TRANSACTION_AMOUNT,
                   TRANSACTION_CATEGORY, TRANSACTION_TYPE, TRANSACTION_REMARK
            FROM TRANSACTIONS WHERE MEMBER_USERNAME = :username
            ORDER BY T_ID
        """),
        ("M5 posts", """
            SELECT P_ID, TITLE, POST_STATUS, SUBMITTED_DATETIME
            FROM POST WHERE MEMBER_USERNAME = :username
            ORDER BY P_ID
        """),
    ]

    for label, sql in checks:
        cursor.execute(sql, {"username": member_username})
        rows = cursor.fetchall()
        print(f"\n{label}: {len(rows)} row(s)")
        for row in rows:
            print(row)

    if analyst_username and analyst_username.strip():
        cursor.execute(
            """SELECT BA_USERNAME, BA_NAME, EMAIL
               FROM BANK_ANALYST WHERE BA_USERNAME = :username""",
            {"username": analyst_username.strip()},
        )
        rows = cursor.fetchall()
        print(f"\nBA1 analyst: {len(rows)} row(s)")
        for row in rows:
            print(row)


# After completing the UI demonstration, run for example:
# show_demo_evidence("demo_member", "demo_analyst")


In [ ]:
# Run this only when the demonstration/testing session is finished.
# cursor.close()
# conn.close()
